# Grounding Evaluation Tutorial

This notebook demonstrates how to evaluate the **grounding** of research outputs - measuring how well the claims in a report are supported by the provided sources.

**What you'll learn:**
- How grounding evaluation works
- Extracting and verifying claims
- Interpreting grounding metrics
- Using grounding eval to detect hallucinations

## Setup

First, let's install dependencies and set up API keys.

In [1]:
%pip install -q tavily-agent-toolkit

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("TAVILY_API_KEY"):
    os.environ["TAVILY_API_KEY"] = getpass.getpass("TAVILY_API_KEY:\n")

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OPENAI_API_KEY:\n")

In [ ]:
# Import the evaluation framework
from tavily_agent_toolkit import ModelConfig, ModelObject, search_and_answer
from tavily_agent_toolkit.evals import (
    compute_grounding_metrics,
    GroundingResult,
)

# Configure the judge model (a smaller, faster model is fine for evaluation)
judge_config = ModelConfig(
    model=ModelObject(
        model="gpt-5-mini",
        api_key=os.environ["OPENAI_API_KEY"],
    ),
)

## Example 1: Evaluating a Well-Grounded Report

Let's start with a report that is well-supported by its sources.

In [7]:
# A well-grounded report with proper citations
report = """
NVIDIA reported record revenue of $35.1 billion in Q3 2024, representing a 94% increase 
year-over-year [1]. The data center segment was the primary driver, contributing $30.8 billion 
to total revenue [1]. 

The company's AI chips continue to dominate the market, with the H100 GPU being widely adopted 
by major cloud providers and AI companies [2].
"""

sources = [
    {
        "url": "https://investor.nvidia.com/q3-2024",
        "title": "NVIDIA Q3 2024 Financial Results",
        "content": "NVIDIA announced quarterly revenue of $35.1 billion for Q3 fiscal 2024, up 94% from a year ago. Data Center revenue was a record $30.8 billion, up 112% from a year ago."
    },
    {
        "url": "https://techcrunch.com/nvidia-ai-dominance",
        "title": "NVIDIA's AI Chip Dominance",
        "content": "NVIDIA's H100 GPU has become the de facto standard for AI training, with adoption across AWS, Google Cloud, Microsoft Azure, and leading AI companies like OpenAI."
    }
]

In [10]:
# Run grounding evaluation
result = await compute_grounding_metrics(
    report=report,
    sources=sources,
    judge_model_config=judge_config,
)

print(f"Grounding Ratio: {result.grounding_ratio:.1%}")
print(f"Citation Accuracy: {result.citation_accuracy:.1%}")
print(f"Total Claims: {result.total_claims}")
print(f"Supported Claims: {result.supported_claims_count}")
print(f"Unsupported Claims: {result.unsupported_claims_count}")

Grounding Ratio: 80.0%
Citation Accuracy: 33.3%
Total Claims: 5
Supported Claims: 4
Unsupported Claims: 1


In [11]:
# View claim details
print("\nClaim Details:")
for claim in result.claim_details:
    status = "Supported" if claim.is_supported else "UNSUPPORTED"
    print(f"  [{status}] {claim.text[:80]}...")
    if claim.source_url:
        print(f"            Source: {claim.source_url}")


Claim Details:
  [UNSUPPORTED] NVIDIA reported record revenue of $35.1 billion in Q3 2024....
  [Supported] NVIDIA's Q3 2024 revenue represented a 94% year-over-year increase....
            Source: https://investor.nvidia.com/q3-2024
  [Supported] The data center segment contributed $30.8 billion to NVIDIA's total revenue in Q...
            Source: https://investor.nvidia.com/q3-2024
  [Supported] The data center segment was the primary driver of NVIDIA's revenue in Q3 2024....
            Source: https://investor.nvidia.com/q3-2024
  [Supported] The NVIDIA H100 GPU is widely adopted by major cloud providers and AI companies....
            Source: https://techcrunch.com/nvidia-ai-dominance


## Example 2: Detecting Hallucinations

Now let's evaluate a report with some unsupported claims (hallucinations).

In [12]:
# A report with some hallucinated content
report_with_hallucinations = """
NVIDIA reported record revenue of $35.1 billion in Q3 2024 [1]. The company announced 
plans to acquire AMD for $50 billion, which would create the world's largest chip company.

CEO Jensen Huang stated that NVIDIA expects to reach $100 billion in annual revenue by 2025,
and the company is developing a new quantum computing chip called the Q1000.
"""

# Same sources as before - note they don't support the acquisition or quantum claims
result_hallucinated = await compute_grounding_metrics(
    report=report_with_hallucinations,
    sources=sources,
    judge_model_config=judge_config,
)

print(f"Grounding Ratio: {result_hallucinated.grounding_ratio:.1%}")
print(f"Unsupported Claims: {result_hallucinated.unsupported_claims_count}")

print("\nUnsupported Claims (Potential Hallucinations):")
for claim in result_hallucinated.claim_details:
    if not claim.is_supported:
        print(f"  - {claim.text}")

Grounding Ratio: 0.0%
Unsupported Claims: 6

Unsupported Claims (Potential Hallucinations):
  - NVIDIA reported record revenue of $35.1 billion in Q3 2024.
  - NVIDIA announced plans to acquire AMD for $50 billion.
  - The proposed acquisition of AMD by NVIDIA would create the world's largest chip company.
  - Jensen Huang is CEO of NVIDIA.
  - Jensen Huang stated that NVIDIA expects to reach $100 billion in annual revenue by 2025.
  - NVIDIA is developing a new quantum computing chip called the Q1000.


## Example 3: Real-World Evaluation with Tavily Search

Let's run a real search and evaluate the grounding of the generated answer.

In [ ]:
# Run a search with answer generation
search_result = await search_and_answer(
    query="What are the latest developments in quantum computing?",
    api_key=os.environ["TAVILY_API_KEY"],
    model_config=ModelConfig(
        model=ModelObject(model="gpt-5-mini", api_key=os.environ["OPENAI_API_KEY"]),
    ),
    max_number_of_subqueries=1,
    max_results=3,
)

print("Generated Answer:")
print(search_result)

Generated Answer:
{'results': [{'url': 'https://arxiv.org/html/2601.02183v1', 'title': 'Developments in superconducting erasure qubits for ...', 'content': 'Quantum computers are inherently noisy, and a crucial challenge for achieving large-scale, fault-tolerant quantum computing is to implement quantum error correction. A promising direction that has made rapid recent progress is to design hardware that has a specific noise profile, leading to a significantly higher threshold for noise with certain quantum error correcting codes. This Perspective focuses on erasure qubits, which enable hardware-efficient quantum error correction, by concatenating an inner code built-in to the hardware with an outer code. We focus on implementations of dual-rail encoded erasure qubits using superconducting qubits, giving an overview of recent developments in theory and simulation, and hardware demonstrators. We also discuss the differences between [...] when building hardware for quantum error-correcti

In [ ]:
# Evaluate the grounding of the answer
grounding_result = await compute_grounding_metrics(
    report=search_result["answer"],
    sources=search_result["results"],
    judge_model_config=judge_config,
)

print("\n" + "="*60)
print(f"Grounding Ratio: {grounding_result.grounding_ratio:.1%}")
print(f"Citation Accuracy: {grounding_result.citation_accuracy:.1%}")
print(f"Claims: {grounding_result.supported_claims_count}/{grounding_result.total_claims} supported")
print("="*60)

## Understanding the Metrics

### Grounding Ratio
The percentage of factual claims that are supported by at least one source.
- **0.9+**: Excellent - almost all claims are well-sourced
- **0.7-0.9**: Good - most claims supported, some may need verification
- **0.5-0.7**: Fair - significant unsupported content
- **<0.5**: Poor - majority of claims lack source support

### Citation Accuracy
Of the citations that exist in the report, what percentage actually support their claims.
- High citation accuracy with low grounding ratio = missing citations
- Low citation accuracy = incorrect citations (worse than no citations)

### Unsupported Claims
The raw count of claims without source support - useful for identifying specific hallucinations.

## Best Practices

1. **Use a deterministic judge model** (temperature=0) for consistent evaluation
2. **Evaluate iteratively** - use grounding metrics during development to improve prompts
3. **Set thresholds** based on your use case:
   - Medical/legal content: require 0.95+ grounding
   - General research: 0.8+ is typically acceptable
   - Creative content: lower thresholds may be appropriate

In [ ]:
# Export results for further analysis
print(grounding_result.to_dict())